Backend

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import requests

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],      # For development only
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SearchRequest(BaseModel):
    startdate: str
    enddate: str
    stocklist: str
    top: int = 20
    market: str = "nasdaq"
        


from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi import Request

templates = Jinja2Templates(directory="templates")
@app.get("/", response_class=HTMLResponse)
def dashboard(request: Request):

    return templates.TemplateResponse(
        "open interest.html",
        {
            "request": request
        }
    )


@app.post("/search")
def search(request: SearchRequest):
    
    print("startdate =", request.startdate, type(request.startdate))
    print("enddate   =", request.enddate, type(request.enddate))
    print("stocklist =", request.stocklist)
    
    try:
        df = run_option_dashboard_query(
        startdate=request.startdate,
        enddate=request.enddate,
        stocklist=request.stocklist,
        market=request.market,
        top=request.top
    )
        dashboard = build_dashboard_data(df)
        
    except RuntimeError as e:
        print(e)
        dashboard = build_dashboard_data(df)
        
    return dashboard

In [2]:
def query_option_oi(
    stockcode,
    targetmonth
):
    """
    Query the option open-interest API.

    Parameters
    ----------
    stockcode : str
        Underlying stock code, e.g. "00388".

    targetmonth : str
        Option month in YYYYMM format, e.g. "202608".

    Returns
    -------
    dict
        Raw JSON response from the OI API.
    """

    url = "https://bithkex.com/api/"

    params = {
        "q": "oisumquery",
        "sum": "open_int",
        "targetmonth": targetmonth,
        "stockcode": stockcode,
        "changetype": "per",
        "compress": "0",
        "format": "json",
        "secret": "54309e61afcec341aaf3636f1f5f57b0a537b9ce"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    result = response.json()

    if result.get("code") != 0:
        raise RuntimeError(
            f"OI API Error: "
            f"{result.get('msg', 'Unknown error')}"
        )

    return result

In [3]:
def option_oi_dataframe(result, targetmonth):
    """
    Convert the OI API response into a standardized DataFrame.

    Parameters
    ----------
    result : dict
        Raw response returned by query_option_oi().

    targetmonth : str
        Target option month in YYYYMM format.

    Returns
    -------
    pd.DataFrame
        Standardized option OI data.
    """

    records = result.get("res", [])

    if not records:
        return pd.DataFrame(
            columns=[
                "Stock Code",
                "Type",
                "Strike",
                "Open Interest",
                "Target Month"
            ]
        )

    df = pd.DataFrame(records)

    # Rename API fields
    df.rename(
        columns={
            "stock_code": "Stock Code",
            "call_type": "Type",
            "strike": "Strike",
            "sum(open_int)": "Open Interest"
        },
        inplace=True
    )

    # Add target month because the API response itself
    # does not contain expiry information.
    df["Target Month"] = targetmonth

    # Make sure numeric fields have the correct type
    df["Strike"] = pd.to_numeric(
        df["Strike"],
        errors="coerce"
    )

    df["Open Interest"] = pd.to_numeric(
        df["Open Interest"],
        errors="coerce"
    )

    return df

In [4]:
def calculate_max_pain(oi_df):
    """
    Calculate the max-pain profile from strike-level option OI.

    Parameters
    ----------
    oi_df : pd.DataFrame
        Output of option_oi_dataframe().

    Returns
    -------
    profile_df : pd.DataFrame
        Profit/payout profile for every candidate settlement price.

    max_pain_strike : float
        Strike with the minimum total option payout.
    """

    if oi_df.empty:
        return pd.DataFrame(), None

    # Make sure the required columns exist
    required_columns = {
        "Type",
        "Strike",
        "Open Interest"
    }

    missing = required_columns - set(oi_df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    # Separate calls and puts
    calls = oi_df[
        oi_df["Type"].str.upper() == "C"
    ].copy()

    puts = oi_df[
        oi_df["Type"].str.upper() == "P"
    ].copy()

    # Candidate settlement prices.
    # We use all available strikes, not just call strikes.
    strikes = sorted(
        oi_df["Strike"].dropna().unique()
    )

    profile = []

    for settlement in strikes:

        call_payout = (
            (settlement - calls["Strike"])
            .clip(lower=0)
            * calls["Open Interest"]
        ).sum()

        put_payout = (
            (puts["Strike"] - settlement)
            .clip(lower=0)
            * puts["Open Interest"]
        ).sum()

        total_payout = (
            call_payout +
            put_payout
        )

        profile.append({
            "Settlement Price": settlement,
            "Call Payout": call_payout,
            "Put Payout": put_payout,
            "Total Payout": total_payout
        })

    profile_df = pd.DataFrame(profile)

    # Find the strike where total payout is minimized
    min_payout = profile_df["Total Payout"].min()

    max_pain_rows = profile_df[
        profile_df["Total Payout"] == min_payout
    ]

    # If there is a tie, take the lowest strike
    max_pain_strike = (
        max_pain_rows["Settlement Price"]
        .min()
    )

    return profile_df, max_pain_strike

In [5]:
# ==========================================================
# Complete Query Pipeline
# ==========================================================

def run_option_dashboard_query(
    startdate,
    enddate,
    stocklist,
    market="nasdaq",
    top=20
):

    # ==========================================================
    # 1. Query option turnover API
    # ==========================================================

    result = query_option_turnover(
        startdate=startdate,
        enddate=enddate,
        stocklist=stocklist,
        market=market,
        top=top
    )


    # ==========================================================
    # 2. Convert JSON to DataFrame
    # ==========================================================

    raw_df = option_dataframe(result)


    if raw_df.empty:
        return pd.DataFrame()


    # ==========================================================
    # 3. Prepare / rename fields
    # ==========================================================

    df = raw_df.copy()


    rename = {

        "option_code": "Ticker",

        "stock_code": "Stock Code",

        "formatted_expiry": "Expiry",

        "call_type": "Type",

        "strike": "Strike",

        "vol": "Volume",

        "open_int": "Open Interest",

        "iv": "IV",

        "delta": "Delta",

        "gamma": "Gamma",

        "theta": "Theta",

        "vega": "Vega",

        "underlying_price_1545": "Underlying Price"

    }


    df.rename(
        columns=rename,
        inplace=True
    )


    # ==========================================================
    # 4. Convert numeric fields where available
    # ==========================================================

    numeric_columns = [

        "Strike",

        "Volume",

        "Open Interest",

        "IV",

        "Delta",

        "Gamma",

        "Theta",

        "Vega",

        "last",

        "bid",

        "ask",

        "chg"

    ]


    for col in numeric_columns:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )


    # ==========================================================
    # 5. Expiry
    # ==========================================================

    if "Expiry" in df.columns:

        df["Expiry"] = pd.to_datetime(
            df["Expiry"],
            format="%Y%m%d",
            errors="coerce"
        )


        df["DTE"] = (
            df["Expiry"]
            -
            pd.Timestamp.today().normalize()
        ).dt.days


        df["TargetMonth"] = (
            df["Expiry"]
            .dt.strftime("%Y%m")
        )

    else:

        df["DTE"] = pd.NA

        df["TargetMonth"] = pd.NA


    # ==========================================================
    # 6. Moneyness
    #
    # Only calculate when underlying price exists.
    # ==========================================================

    if (
        "Underlying Price" in df.columns
        and "Strike" in df.columns
    ):

        df["Moneyness"] = (
            df["Underlying Price"]
            /
            df["Strike"]
        )

    else:

        df["Moneyness"] = pd.NA


    # ==========================================================
    # 7. ITM / ATM / OTM
    #
    # Only calculate when underlying price exists.
    # ==========================================================

    if "Underlying Price" in df.columns:

        df["Status"] = df.apply(
            option_status,
            axis=1
        )

    else:

        df["Status"] = pd.NA


    # ==========================================================
    # 8. Volume / OI
    #
    # Only calculate when Volume exists.
    # ==========================================================

    if (
        "Volume" in df.columns
        and "Open Interest" in df.columns
    ):

        df["Vol/OI"] = (
            df["Volume"]
            /
            df["Open Interest"].replace(
                0,
                pd.NA
            )
        ).round(2)

    else:

        df["Vol/OI"] = pd.NA


    # ==========================================================
    # 9. Query OI API
    # ==========================================================

    oi_frames = []


    if (
        "Stock Code" in df.columns
        and "TargetMonth" in df.columns
    ):

        unique_combinations = (
            df[
                [
                    "Stock Code",
                    "TargetMonth"
                ]
            ]
            .drop_duplicates()
        )


        for _, row in unique_combinations.iterrows():

            stockcode = str(
                row["Stock Code"]
            )

            targetmonth = str(
                row["TargetMonth"]
            )


            try:

                oi_result = query_open_interest_summary(
                    targetmonth=targetmonth,
                    stockcode=stockcode,
                    market=market,
                    changetype="per"
                )


                oi_temp = open_interest_dataframe(
                    oi_result
                )


                if not oi_temp.empty:

                    oi_temp["TargetMonth"] = (
                        targetmonth
                    )

                    oi_frames.append(
                        oi_temp
                    )


            except Exception as e:

                print(
                    f"OI API failed: "
                    f"{market} / "
                    f"{stockcode} / "
                    f"{targetmonth}: {e}"
                )


    # ==========================================================
    # 10. Merge OI API data
    # ==========================================================

    if oi_frames:

        oi_df = pd.concat(
            oi_frames,
            ignore_index=True
        )


        oi_df.rename(
            columns={
                "call_type": "Type"
            },
            inplace=True
        )


        oi_df["stock_code"] = (
            oi_df["stock_code"]
            .astype(str)
        )


        oi_df["Strike"] = pd.to_numeric(
            oi_df["Strike"],
            errors="coerce"
        )


        df["Stock Code"] = (
            df["Stock Code"]
            .astype(str)
        )


        df["Strike"] = pd.to_numeric(
            df["Strike"],
            errors="coerce"
        )


        df = df.merge(

            oi_df[
                [
                    "stock_code",
                    "TargetMonth",
                    "Type",
                    "Strike",
                    "OI_API"
                ]
            ],

            left_on=[
                "Stock Code",
                "TargetMonth",
                "Type",
                "Strike"
            ],

            right_on=[
                "stock_code",
                "TargetMonth",
                "Type",
                "Strike"
            ],

            how="left"
        )


        df.drop(
            columns=["stock_code"],
            inplace=True,
            errors="ignore"
        )


    else:

        df["OI_API"] = pd.NA


    # ==========================================================
    # 11. Sort
    # ==========================================================

    if "Open Interest" in df.columns:

        df = df.sort_values(
            "Open Interest",
            ascending=False
        )


    # ==========================================================
    # 12. Display columns
    # ==========================================================

    display_columns = [

        "Ticker",

        "Stock Code",

        "Expiry",

        "DTE",

        "Type",

        "Strike",

        "Underlying Price",

        "Status",

        "bid",

        "ask",

        "last",

        "chg",

        "Volume",

        "Open Interest",

        "OI_API",

        "Vol/OI",

        "IV",

        "Delta"

    ]


    # Only retain columns actually available
    display_columns = [

        col
        for col in display_columns
        if col in df.columns

    ]


    df = df[display_columns]


    # ==========================================================
    # 13. Format expiry for dashboard
    # ==========================================================

    if "Expiry" in df.columns:

        df["Expiry"] = (
            df["Expiry"]
            .dt.strftime("%Y-%m-%d")
        )


    return df

In [6]:
import pandas as pd

# ==========================================================
# Open Interest Summary API
# ==========================================================
def run_option_dashboard_query(
    startdate,
    enddate,
    stocklist,
    market="nasdaq",
    top=20
):

    # ==========================================================
    # 1. Normalize market
    # ==========================================================

    market = market.lower().strip()

    if market not in {"nasdaq", "hkex"}:
        raise ValueError(
            f"Unsupported market: {market}"
        )


    # ==========================================================
    # 2. Query option turnover API
    # ==========================================================

    result = query_option_turnover(
        startdate=startdate,
        enddate=enddate,
        stocklist=stocklist,
        market=market,
        top=top
    )

    raw_df = option_dataframe(result)


    # ==========================================================
    # 3. Common dashboard columns
    # ==========================================================

    common_columns = [
        "Ticker",
        "Expiry",
        "DTE",
        "Type",
        "Strike",
        "last",
        "Open Interest",
        "IV"
    ]


    # ==========================================================
    # 4. Handle empty result
    # ==========================================================

    if raw_df.empty:

        return pd.DataFrame(
            columns=common_columns
        )


    # ==========================================================
    # 5. Copy raw data
    # ==========================================================

    df = raw_df.copy()


    # ==========================================================
    # 6. Rename common fields
    # ==========================================================

    rename = {

        "option_code": "Ticker",

        "formatted_expiry": "Expiry",

        "call_type": "Type",

        "strike": "Strike",

        "last": "last",

        "iv": "IV",

        "open_int": "Open Interest"

    }

    df.rename(
        columns=rename,
        inplace=True
    )


    # ==========================================================
    # 7. Convert numeric fields
    # ==========================================================

    for column in [
        "Strike",
        "last",
        "IV",
        "Open Interest"
    ]:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

        else:

            df[column] = pd.NA


    # ==========================================================
    # 8. Convert expiry
    # ==========================================================

    df["Expiry"] = pd.to_datetime(
        df["Expiry"],
        format="%Y%m%d",
        errors="coerce"
    )


    # ==========================================================
    # 9. Calculate DTE
    # ==========================================================

    today = pd.Timestamp.today().normalize()

    df["DTE"] = (
        df["Expiry"] - today
    ).dt.days


    # ==========================================================
    # 10. Keep ONLY common dashboard fields
    # ==========================================================

    for column in common_columns:

        if column not in df.columns:

            df[column] = pd.NA


    df = df[
        common_columns
    ].copy()


    # ==========================================================
    # 11. Sort by Open Interest
    # ==========================================================

    df = df.sort_values(
        "Open Interest",
        ascending=False,
        na_position="last"
    )


    # ==========================================================
    # 12. Format expiry
    # ==========================================================

    df["Expiry"] = (
        df["Expiry"]
        .dt.strftime("%Y-%m-%d")
    )


    # ==========================================================
    # 13. Reset index
    # ==========================================================

    df.reset_index(
        drop=True,
        inplace=True
    )


    return df

In [7]:
import requests
import pandas as pd


def query_open_interest(
    stock_code,
    target_month,
    api_key,
    timeout=15
):
    """
    Query Open Interest by strike and option type
    using the BithKEX oisumquery API.

    Parameters
    ----------
    stock_code : str
        HKEX stock code, e.g. "00388".

    target_month : str
        Target option month in YYYYMM format, e.g. "202608".

    api_key : str
        BithKEX API key.

    timeout : int
        HTTP request timeout in seconds.

    Returns
    -------
    dict
        Raw JSON response from the API.
    """

    url = "https://bithkex.com/api/"

    params = {
        "q": "oisumquery",
        "sum": "open_int",
        "targetmonth": str(target_month),
        "stockcode": str(stock_code).zfill(5),
        "market": "hkex",
        "changetype": "per",
        "compress": "0",
        "format": "json",
        "secret": api_key
    }
    

    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    # HTTP-level error
    response.raise_for_status()

    data = response.json()

    # API-level error
    if data.get("code") != 0:
        raise RuntimeError(
            f"Open Interest API error: "
            f"{data.get('msg', 'Unknown error')}"
        )

    # Check that the expected result exists
    if "res" not in data:
        raise RuntimeError(
            "Open Interest API returned no 'res' field."
        )

    return data

In [8]:
def open_interest_dataframe(api_result):
    """
    Convert the oisumquery API response into a clean DataFrame.
    """

    rows = api_result.get("res", [])

    if not rows:
        return pd.DataFrame(
            columns=[
                "Stock Code",
                "Type",
                "Strike",
                "Open Interest",
                "Target Month"
            ]
        )

    df = pd.DataFrame(rows)

    # Rename API fields
    df.rename(
        columns={
            "stock_code": "Stock Code",
            "call_type": "Type",
            "strike": "Strike",
            "sum(open_int)": "Open Interest"
        },
        inplace=True
    )

    # Add target month because the API response itself
    # does not contain expiry information.
    df["Target Month"] = api_result["args"]["targetmonth"]

    # Make sure numeric columns are numeric
    df["Strike"] = pd.to_numeric(
        df["Strike"],
        errors="coerce"
    )

    df["Open Interest"] = pd.to_numeric(
        df["Open Interest"],
        errors="coerce"
    )

    return df[
        [
            "Stock Code",
            "Type",
            "Strike",
            "Open Interest",
            "Target Month"
        ]
    ]

In [9]:
def open_interest_dataframe(api_result):
    """
    Convert the oisumquery API response into a clean DataFrame.
    """

    rows = api_result.get("res", [])

    if not rows:
        return pd.DataFrame(
            columns=[
                "Stock Code",
                "Type",
                "Strike",
                "Open Interest",
                "Target Month"
            ]
        )

    df = pd.DataFrame(rows)

    # Rename API fields
    df.rename(
        columns={
            "stock_code": "Stock Code",
            "call_type": "Type",
            "strike": "Strike",
            "sum(open_int)": "Open Interest"
        },
        inplace=True
    )

    # Add target month because the API response itself
    # does not contain expiry information.
    df["Target Month"] = api_result["args"]["targetmonth"]

    # Make sure numeric columns are numeric
    df["Strike"] = pd.to_numeric(
        df["Strike"],
        errors="coerce"
    )

    df["Open Interest"] = pd.to_numeric(
        df["Open Interest"],
        errors="coerce"
    )

    return df[
        [
            "Stock Code",
            "Type",
            "Strike",
            "Open Interest",
            "Target Month"
        ]
    ]

In [11]:
oi_result = query_open_interest(
    stock_code="00388",
    target_month="202608",
    api_key="54309e61afcec341aaf3636f1f5f57b0a537b9ce"
)

In [12]:
oi_df = open_interest_dataframe(oi_result)

display(oi_df.head(20))
print(oi_df.shape)

,Stock Code,Type,Strike,Open Interest,Target Month
0,00388,P,200,0,202608
1,00388,C,200,59,202608
2,00388,P,205,0,202608
3,00388,C,205,59,202608
4,00388,P,210,0,202608
5,00388,C,210,59,202608
6,00388,P,215,0,202608
7,00388,C,215,0,202608
8,00388,P,220,0,202608
9,00388,C,220,0,202608


(166, 5)


In [18]:
max_pain, payout_df = calculate_max_pain(oi_df)

print("Max Pain:", max_pain)

Max Pain:     Settlement Price  Call Payout  Put Payout  Total Payout
0                200            0     9024075       9024075
1                205          295     8768725       8769020
2                210          885     8513375       8514260
3                215         1770     8258025       8259795
4                220         2655     8002675       8005330
..               ...          ...         ...           ...
78               670     12615645           0      12615645
79               680     13156995           0      13156995
80               690     13698345           0      13698345
81               700     14239695           0      14239695
82               710     14781045           0      14781045

[83 rows x 4 columns]


In [34]:
# ==========================================================
# Prepare dashboard data
# ==========================================================

def prepare_dashboard_dataframe(df):

    df = df.copy()

    rename = {

        "option_code":"Ticker",

        "formatted_expiry":"Expiry",

        "call_type":"Type",

        "strike":"Strike",

        "vol":"Volume",

        "open_int":"Open Interest",

        "iv":"IV",

        "delta":"Delta",

        "gamma":"Gamma",

        "theta":"Theta",

        "vega":"Vega"

    }

    for old, new in rename.items():

        if old in df.columns:

            df.rename(columns={old:new}, inplace=True)

    numeric = [

        "Strike",

        "Volume",

        "Open Interest",

        "IV",

        "Delta",

        "Gamma",

        "Theta",

        "Vega"

    ]

    for col in numeric:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")
            
    df = df.drop(columns=["implied_underlying_price_1545"], errors="ignore")

    return df

In [14]:
def calculate_max_pain(oi_df):
    """
    Calculate the max-pain profile from option open interest.

    Parameters
    ----------
    oi_df : pandas.DataFrame
        Must contain:
            Stock Code
            Type
            Strike
            Open Interest

        Type must contain 'C' and/or 'P'.

    Returns
    -------
    profile_df : pandas.DataFrame
        Columns:
            Settlement Price
            Call Payout
            Put Payout
            Total Payout

    max_pain : numeric
        Settlement price that minimizes total payout.
    """

    required_columns = {
        "Stock Code",
        "Type",
        "Strike",
        "Open Interest"
    }

    missing = required_columns - set(oi_df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    if oi_df.empty:
        return (
            pd.DataFrame(
                columns=[
                    "Settlement Price",
                    "Call Payout",
                    "Put Payout",
                    "Total Payout"
                ]
            ),
            None
        )

    # Work on a copy so the original dataframe is not modified
    df = oi_df.copy()

    # Make sure numerical fields are numeric
    df["Strike"] = pd.to_numeric(
        df["Strike"],
        errors="coerce"
    )

    df["Open Interest"] = pd.to_numeric(
        df["Open Interest"],
        errors="coerce"
    ).fillna(0)

    # Remove invalid rows
    df = df.dropna(subset=["Strike"])

    # Normalize option type
    df["Type"] = (
        df["Type"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Candidate settlement prices
    settlement_prices = sorted(
        df["Strike"].unique()
    )

    profile = []

    for settlement in settlement_prices:

        # -----------------------------
        # Call payout
        # -----------------------------
        calls = df[df["Type"] == "C"]

        call_payout = (
            (
                (settlement - calls["Strike"])
                .clip(lower=0)
            )
            * calls["Open Interest"]
        ).sum()

        # -----------------------------
        # Put payout
        # -----------------------------
        puts = df[df["Type"] == "P"]

        put_payout = (
            (
                (puts["Strike"] - settlement)
                .clip(lower=0)
            )
            * puts["Open Interest"]
        ).sum()

        total_payout = (
            call_payout +
            put_payout
        )

        profile.append({
            "Settlement Price": settlement,
            "Call Payout": call_payout,
            "Put Payout": put_payout,
            "Total Payout": total_payout
        })

    profile_df = pd.DataFrame(profile)

    # Find the settlement price with
    # the lowest aggregate payout
    min_index = profile_df[
        "Total Payout"
    ].idxmin()

    max_pain = profile_df.loc[
        min_index,
        "Settlement Price"
    ]

    return profile_df, max_pain

In [26]:
def prepare_max_pain_data(oi_df):
    """
    Calculate Max Pain and prepare the payout profile
    for dashboard visualization.
    """

    payout_df, max_pain = calculate_max_pain(oi_df)

    payout_df = payout_df.copy()

    payout_df["Is Max Pain"] = (
        payout_df["Settlement Price"] == max_pain
    )

    return {
        "max_pain": max_pain,
        "payout": payout_df
    }

In [30]:
def serialize_max_pain_data(max_pain_data):
    """
    Convert Max Pain calculation results into JSON-safe
    data for the frontend.
    """

    payout_df = max_pain_data["payout"]

    return {
        "max_pain": float(max_pain_data["max_pain"]),
        "payout": payout_df[
            [
                "Settlement Price",
                "Call Payout",
                "Put Payout",
                "Total Payout"
            ]
        ].to_dict(orient="records")
    }

In [19]:
def get_max_pain_data(
    stock_code,
    target_month,
    api_key
):
    """
    Get Open Interest data and calculate Max Pain
    for a single stock and target option month.

    Returns
    -------
    oi_df : DataFrame
        Open Interest by strike and option type.

    payout_df : DataFrame
        Call/Put/Total payout for each settlement price.

    max_pain : numeric
        Settlement price with the minimum total payout.
    """

    # --------------------------------------------------
    # 1. Query Open Interest API
    # --------------------------------------------------

    oi_result = query_open_interest(
        stock_code=stock_code,
        target_month=target_month,
        api_key=api_key
    )

    # --------------------------------------------------
    # 2. Convert API response to DataFrame
    # --------------------------------------------------

    oi_df = open_interest_dataframe(
        oi_result
    )

    # --------------------------------------------------
    # 3. Calculate Max Pain
    # --------------------------------------------------

    payout_df, max_pain = calculate_max_pain(
        oi_df
    )

    return (
        oi_df,
        payout_df,
        max_pain
    )

In [15]:
# ==========================================================
# Dashboard Summary
# ==========================================================

def dashboard_summary(df):

    if len(df) == 0:

        return {

            "tickerCount":0,

            "contractCount":0,

            "maxOI":0,

            "avgIV":0

        }

    summary = {

        "tickerCount":

            df["Ticker"].nunique(),

        "contractCount":

            len(df),

        "maxOI":

            int(df["Open Interest"].max()),

        "avgIV":

            round(df["IV"].mean(),4)

    }

    return summary

In [36]:
# ==========================================================
# Prepare dashboard object
# ==========================================================

def dashboard_data(df):

    summary = dashboard_summary(df)

    rows = df.to_dict(orient="records")

    return {

        "summary":summary,

        "rows":rows

    }

In [37]:
def option_status(row):

    if row["Type"]=="C":

        if row["Underlying Price"] > row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

    else:

        if row["Underlying Price"] < row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

In [38]:
# ==========================================================
# Build Dashboard Data
# ==========================================================

def build_dashboard_data(df):

    summary = {

        "tickerCount": int(df["Ticker"].nunique()),

        "contractCount": int(len(df)),

        "maxOI": int(df["Open Interest"].max()),

        "avgIV": round(float(df["IV"].mean()),3)

    }

    data = {

        "summary": summary,
        
        "columns": df.columns.tolist(),

        "rows": df.to_dict(orient="records")

    }

    return data

In [39]:
import threading
import uvicorn
import pandas as pd

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

INFO:     Started server process [31780]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): 通常每个套接字地址(协议/网络地址/端口)只允许使用一次。
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [40]:
test_df = run_option_dashboard_query(
    startdate="20260801",
    enddate="20260810",
    stocklist="AAPL",
    market="nasdaq",
    top=20
)

display(test_df.head())

,Ticker,Expiry,DTE,Type,Strike,last,Open Interest,IV
0,AAPL,2026-09-18,39,C,320.0,5.80,34208,0.2620
1,AAPL,2026-12-18,130,P,300.0,14.55,13603,0.2809
2,AAPL,2026-10-16,67,P,300.0,12.21,13298,0.2708
3,AAPL,2026-11-20,102,C,310.0,21.22,9998,0.2724
4,AAPL,2026-08-07,-3,P,325.0,22.25,8711,0.3504


In [41]:
test_df = run_option_dashboard_query(
    startdate="20260801",
    enddate="20260810",
    stocklist="00388",
    market="hkex",
    top=20
)

display(test_df.head(20))

,Ticker,Expiry,DTE,Type,Strike,last,Open Interest,IV
0,HEX,2026-09-29,50,C,450,2.86,6526,22
1,HEX,2026-08-28,18,C,420,6.24,6456,20
2,HEX,2026-08-28,18,C,410,11.52,6113,20
3,HEX,2026-08-28,18,C,410,9.19,6113,20
4,HEX,2026-08-28,18,P,420,14.70,5044,22
5,HEX,2026-08-28,18,P,420,11.48,5044,21
6,HEX,2026-08-28,18,C,400,14.90,4100,21
7,HEX,2026-09-29,50,C,390,23.67,2961,20
8,HEX,2026-09-29,50,C,390,22.98,2961,21
9,HEX,2027-06-29,323,C,420,32.47,2320,23


In [43]:
print(test_df.columns.tolist())
display(
    test_df[
        [
            "Ticker",
            "Expiry",
            "Type",
            "Strike",
            "Open Interest",
            "IV",
            "last"
        ]
    ].head(20)
)

['Ticker', 'Expiry', 'DTE', 'Type', 'Strike', 'last', 'Open Interest', 'IV']


,Ticker,Expiry,Type,Strike,Open Interest,IV,last
0,HEX,2026-09-29,C,450,6526,22,2.86
1,HEX,2026-08-28,C,420,6456,20,6.24
2,HEX,2026-08-28,C,410,6113,20,11.52
3,HEX,2026-08-28,C,410,6113,20,9.19
4,HEX,2026-08-28,P,420,5044,22,14.70
5,HEX,2026-08-28,P,420,5044,21,11.48
6,HEX,2026-08-28,C,400,4100,21,14.90
7,HEX,2026-09-29,C,390,2961,20,23.67
8,HEX,2026-09-29,C,390,2961,21,22.98
9,HEX,2027-06-29,C,420,2320,23,32.47


In [ ]:
### result_nasdaq = query_option_turnover(
    startdate="20260801",
    enddate="20260810",
    stocklist="AAPL",
    market="nasdaq",
    top=20
)

raw_nasdaq = option_dataframe(result_nasdaq)

print("NASDAQ columns:")
print(raw_nasdaq.columns.tolist())

print("\nFirst row:")
print(raw_nasdaq.iloc[0].to_dict())

Old version, static dashboard (Abandoned)

Create html

In [3]:
import requests
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

# ==========================================================
# Configuration
# ==========================================================

SECRET = "61f2fd18f1f0634195d6fd07a147f8af870289b4"

BASE_URL = "https://bithkex.com/api/"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def query_option_oi(
    stockcode,
    targetmonth
):
    """
    Query the option open-interest API.

    Parameters
    ----------
    stockcode : str
        Underlying stock code, e.g. "00388".

    targetmonth : str
        Option month in YYYYMM format, e.g. "202608".

    Returns
    -------
    dict
        Raw JSON response from the OI API.
    """

    url = "https://bithkex.com/api/"

    params = {
        "q": "oisumquery",
        "sum": "open_int",
        "targetmonth": targetmonth,
        "stockcode": stockcode,
        "changetype": "per",
        "compress": "0",
        "format": "json",
        "secret": "54309e61afcec341aaf3636f1f5f57b0a537b9ce"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    result = response.json()

    if result.get("code") != 0:
        raise RuntimeError(
            f"OI API Error: "
            f"{result.get('msg', 'Unknown error')}"
        )

    return result

In [4]:
# ==========================================================
# Convert API response to DataFrame
# ==========================================================

def option_dataframe(api_result):

    if "res" not in api_result:

        return pd.DataFrame()

    df = pd.DataFrame(api_result["res"])

    return df

In [5]:
# ==========================================================
# Prepare dashboard data
# ==========================================================

def prepare_dashboard_dataframe(df):

    df = df.copy()

    rename = {

        "option_code":"Ticker",

        "formatted_expiry":"Expiry",

        "call_type":"Type",

        "strike":"Strike",

        "vol":"Volume",

        "open_int":"Open Interest",

        "iv":"IV",

        "delta":"Delta",

        "gamma":"Gamma",

        "theta":"Theta",

        "vega":"Vega"

    }

    for old, new in rename.items():

        if old in df.columns:

            df.rename(columns={old:new}, inplace=True)

    numeric = [

        "Strike",

        "Volume",

        "Open Interest",

        "IV",

        "Delta",

        "Gamma",

        "Theta",

        "Vega"

    ]

    for col in numeric:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")
            
    df = df.drop(columns=["implied_underlying_price_1545"], errors="ignore")

    return df

In [6]:
# ==========================================================
# Dashboard Summary
# ==========================================================

def dashboard_summary(df):

    if len(df) == 0:

        return {

            "tickerCount":0,

            "contractCount":0,

            "maxOI":0,

            "avgIV":0

        }

    summary = {

        "tickerCount":

            df["Ticker"].nunique(),

        "contractCount":

            len(df),

        "maxOI":

            int(df["Open Interest"].max()),

        "avgIV":

            round(df["IV"].mean(),4)

    }

    return summary

In [7]:
# ==========================================================
# Prepare dashboard object
# ==========================================================

def dashboard_data(df):

    summary = dashboard_summary(df)

    rows = df.to_dict(orient="records")

    return {

        "summary":summary,

        "rows":rows

    }

In [8]:
# Query API
result = query_option_turnover(
    startdate="20260627",
    enddate="20260726",
    stocklist="AAPL,MSFT",
    market="nasdaq",
    top=20
)

# Convert to DataFrame
raw_df = option_dataframe(result)

# Prepare dashboard DataFrame
dashboard_df = prepare_dashboard_dataframe(raw_df)

# Build dashboard object
dashboard = dashboard_data(dashboard_df)

print(dashboard["summary"])

display(dashboard_df.head())

{'tickerCount': 2, 'contractCount': 20, 'maxOI': 75299, 'avgIV': 0.6017}


,Ticker,stock_code,expiry,Type,Strike,qdate,Open Interest,last,chg,bid,...,Theta,Vega,rho,vwap,open,high,low,underlying_price_1545,formatted_qdate,Expiry
0,AAPL,AAPL,2026-10-15T16:00:00.000Z,C,360,2026-07-15T16:00:00.000Z,75299,9.20,1.80,8.75,...,-0.0856,0.6037,0.2560,8.1925,7.40,9.85,6.85,332.280,20260716,20261016
1,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-20T16:00:00.000Z,25591,47.25,2.59,46.95,...,-0.0710,0.2759,0.3993,46.3575,44.66,48.85,42.05,327.770,20260721,20260918
2,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-21T16:00:00.000Z,15556,44.00,-1.24,43.60,...,-0.0715,0.2815,0.3896,44.3080,45.24,45.24,43.30,324.445,20260722,20260918
3,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-22T16:00:00.000Z,6050,41.20,0.25,40.85,...,-0.0708,0.3036,0.3722,40.8695,40.95,41.54,40.19,321.115,20260723,20260918
4,AAPL,AAPL,2026-08-20T16:00:00.000Z,C,345,2026-07-15T16:00:00.000Z,41114,7.40,2.25,6.95,...,-0.1521,0.3935,0.1193,6.4856,5.15,8.10,4.80,332.280,20260716,20260821


In [9]:
rename = {

    "option_code":"Ticker",

    "formatted_expiry":"Expiry",

    "call_type":"Type",

    "strike":"Strike",

    "vol":"Volume",

    "open_int":"Open Interest",

    "iv":"IV",

    "delta":"Delta",

    "gamma":"Gamma",

    "theta":"Theta",

    "vega":"Vega",

    "underlying_price_1545":"Underlying Price"

}


dashboard_df.rename(columns=rename, inplace=True)


Additional information and cleaning

In [10]:
def option_status(row):

    if row["Type"]=="C":

        if row["Underlying Price"] > row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

    else:

        if row["Underlying Price"] < row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

In [11]:
today = pd.Timestamp.today().normalize()

dashboard_df["Expiry"] = pd.to_datetime(dashboard_df["Expiry"], format="%Y%m%d")

dashboard_df["DTE"] = (dashboard_df["Expiry"] - today).dt.days

dashboard_df["Moneyness"] = (dashboard_df["Underlying Price"] / dashboard_df["Strike"])

dashboard_df["Status"] = dashboard_df.apply(option_status, axis=1)

dashboard_df["Vol/OI"] = (dashboard_df["Volume"] / dashboard_df["Open Interest"]).round(2)

dashboard_df = dashboard_df.sort_values("Open Interest", ascending=False)

#dashboard_df = dashboard_df.sort_values("Volume", ascending=False)

In [12]:
dashboard_df.head()

,Ticker,stock_code,expiry,Type,Strike,qdate,Open Interest,last,chg,bid,...,open,high,low,Underlying Price,formatted_qdate,Expiry,DTE,Moneyness,Status,Vol/OI
0,AAPL,AAPL,2026-10-15T16:00:00.000Z,C,360,2026-07-15T16:00:00.000Z,75299,9.20,1.80,8.75,...,7.40,9.85,6.85,332.280,20260716,2026-10-16,79,0.923000,OTM,0.03
4,AAPL,AAPL,2026-08-20T16:00:00.000Z,C,345,2026-07-15T16:00:00.000Z,41114,7.40,2.25,6.95,...,5.15,8.10,4.80,332.280,20260716,2026-08-21,23,0.963130,OTM,0.10
1,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-20T16:00:00.000Z,25591,47.25,2.59,46.95,...,44.66,48.85,42.05,327.770,20260721,2026-09-18,51,1.150070,ITM,0.40
15,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,300,2026-07-22T16:00:00.000Z,25546,29.06,-1.24,28.05,...,30.30,31.00,27.85,321.115,20260723,2026-09-18,51,1.070383,ITM,0.02
7,AAPL,AAPL,2026-07-19T16:00:00.000Z,P,335,2026-07-19T16:00:00.000Z,22579,8.40,4.80,6.85,...,3.60,10.70,2.83,328.045,20260720,2026-07-20,-9,0.979239,ITM,0.99


In [13]:
print(dashboard_df.columns.tolist())

['Ticker', 'stock_code', 'expiry', 'Type', 'Strike', 'qdate', 'Open Interest', 'last', 'chg', 'bid', 'ask', 'Volume', 'size', 'mvdiff', 'oidiff', 'IV', 'Delta', 'Gamma', 'Theta', 'Vega', 'rho', 'vwap', 'open', 'high', 'low', 'Underlying Price', 'formatted_qdate', 'Expiry', 'DTE', 'Moneyness', 'Status', 'Vol/OI']


In [14]:
# Remove duplicates

drop_columns = [
    "stock_code",          # same as Ticker
    "expiry",              # duplicate of Expiry
    "qdate",
    "formatted_qdate",
    "size",                # always 100
    "mvdiff",
    "oidiff"
]

dashboard_df.drop(
    columns=drop_columns,
    inplace=True,
    errors="ignore"
)

In [15]:
# What information to display

display_columns = [
    "Ticker",
    "Expiry",
    "DTE",
    "Type",
    "Strike",
    "Underlying Price",
    "Status",
    "bid",
    "ask",
    "last",
    "chg",
    "Volume",
    "Open Interest",
    "Vol/OI",
    "IV",
    "Delta"
]

dashboard_display_df = dashboard_df[display_columns].copy()

dashboard_display_df = dashboard_display_df.copy()

dashboard_display_df["Expiry"] = dashboard_display_df["Expiry"].dt.strftime("%Y-%m-%d")

Build html

In [16]:
# ==========================================================
# Build Dashboard Data
# ==========================================================

def build_dashboard_data(df):

    summary = {

        "tickerCount": int(df["Ticker"].nunique()),

        "contractCount": int(len(df)),

        "maxOI": int(df["Open Interest"].max()),

        "avgIV": round(float(df["IV"].mean()),3)

    }

    data = {

        "summary": summary,
        
        "columns": df.columns.tolist(),

        "rows": df.to_dict(orient="records")

    }

    return data

In [18]:
import json
import webbrowser
from pathlib import Path


def generate_dashboard(
    df,
    template="open interest.html",
    output="open interest dashboard.html"
):

    dashboard = build_dashboard_data(df)

    dashboard_json = json.dumps(
        dashboard,
        default=str,
        indent=2
    )
    
    print(dashboard_json)

    with open(template, "r", encoding="utf-8") as f:
        html = f.read()

    html = html.replace(
        "{{OPTION_DATA}}",
        dashboard_json
    )
    

    with open(output, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Dashboard saved to {output}")

In [20]:
generate_dashboard(dashboard_display_df)

{
  "summary": {
    "tickerCount": 2,
    "contractCount": 20,
    "maxOI": 75299,
    "avgIV": 0.602
  },
  "columns": [
    "Ticker",
    "Expiry",
    "DTE",
    "Type",
    "Strike",
    "Underlying Price",
    "Status",
    "bid",
    "ask",
    "last",
    "chg",
    "Volume",
    "Open Interest",
    "Vol/OI",
    "IV",
    "Delta"
  ],
  "rows": [
    {
      "Ticker": "AAPL",
      "Expiry": "2026-10-16",
      "DTE": 79,
      "Type": "C",
      "Strike": 360,
      "Underlying Price": 332.28,
      "Status": "OTM",
      "bid": 8.75,
      "ask": 9.0,
      "last": 9.2,
      "chg": 1.8,
      "Volume": 2590,
      "Open Interest": 75299,
      "Vol/OI": 0.03,
      "IV": 0.2681,
      "Delta": 0.3272
    },
    {
      "Ticker": "AAPL",
      "Expiry": "2026-08-21",
      "DTE": 23,
      "Type": "C",
      "Strike": 345,
      "Underlying Price": 332.28,
      "Status": "OTM",
      "bid": 6.95,
      "ask": 7.1,
      "last": 7.4,
      "chg": 2.25,
      "Volume": 4184,

FileNotFoundError: [Errno 2] No such file or directory: 'open interest.html'

In [ ]:
dashboard = build_dashboard_data(dashboard_display_df)

print(dashboard.keys())

                    FastAPI /search
                          │
                          ▼
             run_option_dashboard_query()
                          │
                 ┌──────┴──────┐
                 ▼                 ▼
           Option turnover API           OI API
                 │                 │
                 ▼                 ▼
             Option table           OI dataframe
                                   │
                                   ▼
                             calculate_max_pain()
                                   │
                                   ▼
                              Max Pain profile